In [ ]:
import requests
import json

# Replace 'vik-gpt-core' with the name of your Solr core
core_name = "vik-gpt-core"
# Replace with your Solr instance URL
solr_url = f"http://localhost:8983/solr/{core_name}/select"

# The search query
query = "Milyen tárgyat tanít Mészáros Tamás"

# Construct the query parameters using eDisMax
params = {
    "q": query,  # Use the raw query without field specification
    "defType": "edismax",  # Use the Extended DisMax query parser
    "qf": "text_t^10 title_t^15",  # Query fields with weights (adjust as needed)
    "wt": "json",  # Response format (json)
    "rows": 10,    # Number of rows to return
}

# Make the HTTP GET request to Solr
response = requests.get(solr_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON
    response_json = response.json()
    # Extract documents from the response
    documents = response_json.get("response", {}).get("docs", [])
    
    # Print the documents
    print(f"Found {len(documents)} documents matching the query '{query}':")
    # for doc in documents:
    #     print(json.dumps(doc, indent=2))
    _ = [print(i['title_t']) for i in documents]
else:
    print("Failed to query Solr:", response.text)


In [ ]:
path_to_test_set = "evaluation/tad-qa-dataset/one-course-dataset.json"

with open(path_to_test_set, "r", encoding="utf-8") as f:
    test_set = json.load(f)

In [ ]:
import matplotlib.pyplot as plt

# Assuming solr_url is defined and test_set is available
results = {n: {'correct': 0, 'total': 0} for n in range(1, 11)}

for example in test_set:
    context = example["context"]
    title_target = context.split("\n")[1][len("A kurzus címe: "):-1]
    
    for question in example["questions"]:
        query = question["question"]
        params = {
            "q": query,
            "defType": "edismax",
            "qf": "text_t^10 title_t^15",
            "wt": "json",
            "rows": 10,  # Request the maximum rows once
        }
        response = requests.get(solr_url, params=params)
        if response.status_code == 200:
            response_json = response.json()
            documents = response_json.get("response", {}).get("docs", [])
            titles = [doc['title_t'] for doc in documents]
            
            for n in range(1, 11):
                if title_target in titles[:n]:
                    results[n]['correct'] += 1
                results[n]['total'] += 1

# Calculate accuracies and plot
accuracies = [results[n]['correct'] / results[n]['total'] if results[n]['total'] > 0 else 0 for n in range(1, 11)]


In [ ]:
# Creating a new bar chart with preferred color palette (black, white, and blue)
import numpy as np
# Setting up the plot with a white background
plt.figure(figsize=(12, 8), facecolor='white')
n_documents = np.arange(1, 11)  # Array from 1 to 10 for the x-axis

# Creating the bar chart with a blue color palette
plt.bar(n_documents, accuracies, color='midnightblue', edgecolor='black', linewidth=1, alpha=0.8)

# Adding title and labels with a minimalist style
plt.title('Accuracy of Solr Title Matching for Top N Documents', fontsize=18, fontweight='bold', color='black')
plt.xlabel('Top N Documents', fontsize=14, fontweight='bold', color='black')
plt.ylabel('Accuracy', fontsize=14, fontweight='bold', color='black')

# Customizing the ticks to fit the black and blue theme
plt.xticks(n_documents, fontsize=12, fontweight='bold', color='black')
plt.yticks(np.arange(0, 1.1, 0.1), fontsize=12, fontweight='bold', color='black')

# Adding a subtle grid for better readability, keeping it minimal
plt.grid(axis='y', linestyle='--', alpha=0.3, color='grey')

# Adding value labels on top of each bar in blue to match the theme
for i, acc in enumerate(accuracies):
    plt.text(i + 1, acc + 0.02, f"{acc:.2f}", ha='center', va='bottom', fontsize=12, fontweight='bold', color='blue')

# Show plot with tight layout to ensure everything fits nicely
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import faiss  # Ensure faiss is imported
from dotenv import load_dotenv
import os

load_dotenv()

# Access the OpenAI API key from environment variables
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key not found in environment variables")

# If you need to set it as an environment variable for other modules
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

vectorstore_tad = faiss.load_local('../faiss_db/tad-db')


In [ ]:
!pip install faiss-cpu

In [ ]:

# Assuming embeddings for questions can be generated using a function `generate_embedding(question)`
# and `vectorstore_tad` is your FAISS index loaded as follows:
vectorstore_tad = faiss.load_local('../faiss_db/tad-db')

results = {n: {'correct': 0, 'total': 0} for n in range(1, 11)}

for example in test_set:
    context = example["context"]
    title_target = context.split("\n")[1][len("A kurzus címe: "):-1]
    title_target_embedding = generate_embedding(title_target)  # Generate embedding for the target title
    
    for question in example["questions"]:
        query_embedding = generate_embedding(question["question"])  # Convert question to embedding
        D, I = vectorstore_tad.search(query_embedding.reshape(1, -1), 10)  # Retrieve top 10 closest documents
        
        # Assuming you have a mapping or a way to retrieve titles from their embeddings or indices
        retrieved_titles = [get_title_from_index(index) for index in I[0]]
        
        for n in range(1, 11):
            if title_target in retrieved_titles[:n]:
                results[n]['correct'] += 1
            results[n]['total'] += 1

# Calculate accuracies
accuracies = [results[n]['correct'] / results[n]['total'] if results[n]['total'] > 0 else 0 for n in range(1, 11)]

# Plotting code remains the same
plt.figure(figsize=(12, 8), facecolor='white')
n_documents = np.arange(1, 11)  # Array from 1 to 10 for the x-axis
plt.bar(n_documents, accuracies, color='midnightblue', edgecolor='black', linewidth=1, alpha=0.8)
plt.title('Accuracy of FAISS Embedding Title Matching for Top N Documents', fontsize=18, fontweight='bold', color='black')
plt.xlabel('Top N Documents', fontsize=14, fontweight='bold', color='black')
plt.ylabel('Accuracy', fontsize=14, fontweight='bold', color='black')
plt.xticks(n_documents, fontsize=12, fontweight='bold', color='black')
plt.yticks(np.arange(0, 1.1, 0.1), fontsize=12, fontweight='bold', color='black')
plt.grid(axis='y', linestyle='--', alpha=0.3, color='grey')
for i, acc in enumerate(accuracies):
    plt.text(i + 1, acc + 0.02, f"{acc:.2f}", ha='center', va='bottom', fontsize=12, fontweight='bold', color='blue')
plt.tight_layout()
plt.show()
